<h1>Boeing Final Assembly Rollout & Parking Optimization</h1>

<h2>User-Defined Values</h2>
<h3>1. Initialize for local machine</h3>

In [5]:
# user def / filepaths

user = 'jag'

userpaths = {
    'jag': '/Users/jishnughosh/Documents/GitHub/Boeing-Final-Assembly-and-Rollout-Optimization',
    'ksg': '',
    'mpg': '/Users/mpgen/OneDrive - purdue.edu/PURDUE/Spring 26/IE 431 - Personal',
    'da': '',
    'lah': ''
}

rootpath = userpaths[user] # overwrite if necessary

filepaths = {
    'FAstatus': '/input/FA_Status_FGI_Handoff.xlsx',
    'FGI_Locations': '/input/FGI_Locations_wPriority.xlsx',
    'Nodes': '/input/Nodes.xlsx'
}

<h3>2. Assumed Values</h3>
<p><strong>Changeable parameters:</strong></p>
<ul>
    <li><code>FGI_STAFFING_BYSHIFT</code>: Number of employees on each team for each shift</li>
    <li><code>FGI_CPJ</code>: Cost per job for each FGI team (# manhours required to complete 1 BTG task</li>
    <li><code>CENTERLINES</code>: Planes that must be moved for a move to occur starting at each location</li>

In [6]:
# algorithm toggles

FGI_STAFFING_BYSHIFT = { # FGI team: # manhours / 8hr shift
            1: {'structure': 16, 'systems': 60, 'declam': 16, 'test': 18},
            2: {'structure': 6, 'systems': 14, 'declam': 4, 'test': 0},
            3: {'structure': 0, 'systems': 0, 'declam': 0, 'test': 0}
        }

FGI_CPJ = { # FGI team: # manhours consumed / 1 BTG completed
        'structure': 6,
        'systems': 3.5,
        'declam': 6,
        'test': 8
    }

CENTERLINES = {
    'A1': None,
    'A2': None,
    'A3': None,
    'A4': None,
    'A5': None,
    'A6': None,
    'A7': None,
    'A8': None,
    'A9': ['C1'],
    'A10': ['C1', 'C2'],
    'BSC1': ['C1', 'C2', 'C3', 'C3.5', 'C4'],
    'BSC2': ['C1', 'C2', 'C3', 'C3.5', 'C4', 'C5'],
    'C1': None,
    'C2': ['C1'],
    'C3': ['C1', 'C2'],
    'C3.5': ['C1', 'C2', 'C3'],
    'C4': ['C1', 'C2', 'C3', 'C3.5'],
    'C5': ['C1', 'C2', 'C3', 'C3.5', 'C4'],
    'CR1': ['CR3', 'CR2'],
    'CR2': ['CR3'],
    'CR3': None,
    'D1': None,
    'D2': None,
    'F1': ['C1', 'C2'],
    'F2': ['C1', 'C2'],
    'L4': None,
    'L5': ['L4'],
    'Spur': None,
    'T1': None,
    'T2': None,
    'T3': None,
    'T4': None
}



<p>|\==><i>NOTE: THESE PARAMETERS WILL BE OVERWRITTEN IF INPUT_TYPE IS SET TO FGI_LIVESTATE</p>

<h2>Initialization</h2>
<h3>1. Import Libraries</h3>
<p><strong>Required Installations:</strong></p>
<ul>
    <li>pandas</li>
    <li>numpy</li>
    <li>datetime</li>
    <li>matplotlib.pyplot</li>
    <li>seaborn</li>
    <li>math</li>
</ul>

In [7]:
import pandas as pd
import numpy as np
import datetime as dt
from datetime import timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import math
from math import dist
# import ast
# import heapq
# import re
# import os

<h3>2. Run Conditions</h3>
<p>These parameters set the conditions under which the algorithm will generate the schedule. Be sure that this is updated on each run of the main placement algorithm</p>

In [8]:
# a) planning horizon
STARTDATE='2024-06-10'
ENDDATE='2026-03-10'
FORECAST_UNTIL_COMPLETION = True

# b) location parameters

NEW_FA_ONLINE = False
DISCONTINUED_LOCATIONS = ['S1', 'S2', 'S3', 'Spur']
INCLUDE_FA_LOCATIONS = False

# c) schedule parameters
PLANNED_BUFFER_DAYS = []
IMPORT_PAINT_SCHEDULE = False


# d) input parameters
input_types = [
    'FARO_SCORECARD_WITH_MILESTONES',
    'FGI_LIVESTATE'
]
INPUT_TYPE = input_types[0]

# e) output/export parameters
EXPORT_TO_EXCEL = True
EXPORT_PATH = '/output/'
CODECELL_OUTPUT = True
EXPORT_TO_FGI_LIVESTATE = True


<p><i>|==> Initialize based on run conditions </i></p>

In [10]:
# dataframe builder functions
def build_ap_df(faro_scorecard, p3_milestones, tankClosure):
    ap_data = merge_APdata(faro_scorecard, p3_milestones, tankClosure)

    rows = []

    for _, row in ap_data.iterrows():
        ln = int(row['LN'])

        rows.append({
            'LN': ln,
            'FA RO': row['FA RO'],
            'FA RO to B1R': row['FA RO to B1R'],
            'Total Counters': row['Total Counters'],
            'TankStatus': row['TankStatus'],
            'Ceilings': row['Ceilings'],
            'Initial Tests Run': row['Initial Tests Run'],

            # BTG attributes
            'BTG_tot': row['Total BTG'],
            'BTG_p0': row['P0 BTG'],
            'BTG_p1': row['P1 BTG'],
            'BTG_p2': row['P2 BTG'],
            'BTG_p3': row['P3 BTG'],
            'BTG_engines': row['Engines BTG'],
            'BTG_doors': row['Doors BTG'],
            'BTG_test': row['Test BTG'],

            # P3 milestone attributes
            'P3_Engine Hang': row['Engine Hang'],
            'P3_Flight Controls': row['Flight Controls'],
            'P3_Gear Swing': row['Gear Swing'],
            'P3_Medium Pressure Test': row['Medium Pressure Test'],
            'P3_Oil On': row['Oil On'],
            'P3_Power On': row['Power On'],
            'P3_Engine_Type': row['Engine_Type'],
            'P3_Milestone_Completion_Percentage': row['Milestone_Completion_Percentage'],

            # Shake attributes
            'shakes_complete': row['shakes_completed'],
            'shakes_req': row['shakes_req']
        })

    return pd.DataFrame(rows)

def build_location_df(fa_priority, centerlines):
    rows = []
    seen_locations = set()

    for _, row in fa_priority.iterrows():
        loc = row['Location']
        seen_locations.add(loc)
        centerline_deps = centerlines.get(loc, None)
        priority = 99 if loc in {'D1', 'D2'} else row['Future State Priority']

        rows.append({
            'Location': loc,
            'Future State Priority': priority,
            'Date Online': row['Date Online'],
            'Owner': row['Owner'],
            'tooling_jacking': row['Jacking'] == 'Y',
            'tooling_wings': row['Wings'] == 'Y',
            'tooling_tankClosure': row['Tank Closure'] == 'Y',
            'centerline_dependencies': None if centerline_deps is None else ', '.join(centerline_deps),
            'obstructions': None,
            'notes': None
        })

    return pd.DataFrame(rows)

def build_labor_df(fgi_staffing_byshift, fgi_cpj, startdate, enddate):
    rows = []

    # FGI staffing by shift
    for shift, teams in fgi_staffing_byshift.items():
        for team, manhours in teams.items():
            rows.append({
                'category': 'FGI_STAFFING_BYSHIFT',
                'shift': shift,
                'team': team,
                'value': manhours
            })

    # FGI CPJ
    for team, cpj in fgi_cpj.items():
        rows.append({
            'category': 'FGI_CPJ',
            'shift': None,
            'team': team,
            'value': cpj
        })

    # Dates
    rows.append({
        'category': 'CONFIG',
        'shift': None,
        'team': 'STARTDATE',
        'value': startdate
    })
    rows.append({
        'category': 'CONFIG',
        'shift': None,
        'team': 'ENDDATE',
        'value': enddate
    })

    return pd.DataFrame(rows)


# build dataframes from FARO scorecare with milestones
if INPUT_TYPE == 'FARO_SCORECARD_WITH_MILESTONES':
    def clean_FAstatus(faro_scorecard, tankClosure, p3_milestones):
        ## FARO Scorecard Cleaning
        # only take rows with valid LNs
        faro_scorecard = faro_scorecard[pd.to_numeric(faro_scorecard['LN'], errors='coerce').notna()].copy()
        # remove unnamed columns
        faro_scorecard = faro_scorecard.loc[:, ~faro_scorecard.columns.astype(str).str.contains(r'^Unnamed')]

        # split zone shakes column into two columns: completed/required
        faro_scorecard[['shakes_completed', 'shakes_req']] = faro_scorecard['Zone Shakes'].astype(str).str.split('/', expand=True)
        # only take 
        faro_scorecard['p3_milestones'] = faro_scorecard['P3 Milestones'].astype(str).str.split('/').str[0]

        # set datatypes
        faro_scorecard = (
            faro_scorecard.assign(
                LN=lambda df: pd.to_numeric(df['LN'], errors='coerce').astype(int),
                **{
                    'FA RO to B1R': lambda df: pd.to_numeric(df['FA RO to B1R'], errors='coerce'),
                    'Total Counters': lambda df: pd.to_numeric(df['Total Counters'], errors='coerce').fillna(0),
                    'Total BTG': lambda df: pd.to_numeric(df['Total BTG'], errors='coerce').fillna(0),
                    'P0 BTG': lambda df: pd.to_numeric(df['P0 BTG'], errors='coerce').fillna(0),
                    'P1 BTG': lambda df: pd.to_numeric(df['P1 BTG'], errors='coerce').fillna(0),
                    'P2 BTG': lambda df: pd.to_numeric(df['P2 BTG'], errors='coerce').fillna(0),
                    'P3 BTG': lambda df: pd.to_numeric(df['P3 BTG'], errors='coerce').fillna(0),
                    'Engines BTG': lambda df: pd.to_numeric(df['Engines BTG'], errors='coerce').fillna(0),
                    'Doors BTG': lambda df: pd.to_numeric(df['Doors BTG'], errors='coerce').fillna(0),
                    'Test BTG': lambda df: pd.to_numeric(df['Test BTG'], errors='coerce').fillna(0),
                    'Tank Closure': lambda df: df['Tank Closure'].map({'Open': 0, 'Closed': 1}).fillna(0).astype(int),
                    'Ceilings': lambda df: pd.to_numeric(df['Ceilings'], errors='coerce').fillna(0),
                    'Initial Tests Run': lambda df: (
                        df['Initial Tests Run'].astype(str)
                        .str.replace('%', '', regex=False)
                        .replace('nan', 0)
                        .replace('', 0)
                        .astype(float) / 100
                    ),
                    'shakes_completed': lambda df: pd.to_numeric(df['shakes_completed'], errors='coerce').fillna(0).astype(int),
                    'shake_req': lambda df: pd.to_numeric(df['shakes_req'], errors='coerce').fillna(0).astype(int),
                    'p3_milestones': lambda df: pd.to_numeric(df['p3_milestones'], errors='coerce').fillna(0).astype(int),
                }
            )
        )

        ## TANK CLOSURE Cleaning
        tankClosure['LINE_NUMBER'] = pd.to_numeric(tankClosure['LINE_NUMBER'], errors='coerce')
        tankClosure['Complete_Jobs'] = pd.to_numeric(tankClosure['Complete_Jobs'], errors='coerce').fillna(0)
        tankClosure['Total_Jobs'] = pd.to_numeric(tankClosure['Total_Jobs'], errors='coerce').fillna(0)
        tankClosure['TankStatus'] = tankClosure['TankStatus'].map({'Open': 0, 'Closed': 1}).fillna(0).astype(int)

        ## P3 Milestone Cleaning
        p3_milestones['Engine_Type'] = p3_milestones['Milestone'].astype(str).str.extract(r'\((.*?)\)')
        p3_milestones['Milestone'] = p3_milestones['Milestone'].astype(str).str.replace(r'\s*\(.*?\)', '', regex=True)
        p3_milestones = (
            p3_milestones.pivot_table(index='P', columns='Milestone', values='Completed_Jobs', aggfunc='sum')
            .assign(
                Engine_Type=p3_milestones.groupby('P')['Engine_Type'].first(),
                Milestone_Completion_Percentage=p3_milestones.groupby('P')['STATUS (1 Complete, 0 Still Open)'].mean()
            )
            .fillna(-1)
            .reset_index()
        )

        return faro_scorecard, tankClosure, p3_milestones

    def clean_nodeData(nodes, node_adj):
        ## clean nodes
        nodes = (
            nodes
            .drop(columns=[c for c in nodes.columns if str(c).startswith('Unnamed')], errors='ignore')
            .dropna(how='all')
            .assign(
                node_id=lambda df: df['node_id'].astype('string').str.strip(),
                x=lambda df: pd.to_numeric(df['x'], errors='coerce'),
                y=lambda df: pd.to_numeric(df['y'], errors='coerce'),
                type=lambda df: df['type'].astype('string').str.strip(),
                req_centerline=lambda df: df['req_centerline'].astype('string').str.strip()
            )
            .replace({'': pd.NA})
            .dropna(subset=['node_id', 'x', 'y'])
            .reset_index(drop=True)
        )

        ## clean node_adj 
        node_adj = (
            node_adj
            .drop(columns=[c for c in node_adj.columns if str(c).startswith('Unnamed')], errors='ignore')
            .dropna(how='all')
            .assign(
                from_node=lambda df: df['from_node'].astype('string').str.strip(),
                to_node=lambda df: df['to_node'].astype('string').str.strip()
            )
            .replace({'': pd.NA})
            .dropna(subset=['from_node', 'to_node'])
            .drop_duplicates()
            .reset_index(drop=True)
        )

        return nodes, node_adj

    def import_data(rootpath, filepaths = {'FAstatus': 'FA_Status_FGI_Handoff.xlsx','FGI_Locations': 'FGI_Locations_wPriority.xlsx','Nodes': 'Nodes.xlsx'}):
        
        ## FA STATUS DATA IMPORT
        path = rootpath + filepaths['FAstatus']
        # read FARO scorecard as df
        faro_scorecard = pd.read_excel(path, sheet_name='FARO_Scorecard',header=2,engine='openpyxl')
        tankClosure = pd.read_excel(path,sheet_name='Tank_Closure_Detail',engine='openpyxl')
        p3_milestones = pd.read_excel(path, sheet_name='P3 Milestone Detail',engine='openpyxl')

        ## FGI LOCATION DATA IMPORT
        path = rootpath + filepaths['FGI_Locations']
        fa_priority = pd.read_excel(path, sheet_name='FA Priority', header=1,engine='openpyxl')

        ## NODES DATA IMPORT
        path = rootpath + filepaths['Nodes']
        nodes = pd.read_excel(path,sheet_name='Node',engine='openpyxl')
        node_adj = pd.read_excel(path, sheet_name='adjacency',engine='openpyxl')

        ## DATA CLEANING
        faro_scorecard, tankClosure, p3_milestones = clean_FAstatus(faro_scorecard, tankClosure, p3_milestones)

        fa_priority = (
            fa_priority
            .drop(columns=[c for c in fa_priority.columns if str(c).startswith('Unnamed')], errors='ignore')
            .dropna(how='all')
            .reset_index(drop=True)
        )

        nodes, node_adj = clean_nodeData(nodes, node_adj)

        return faro_scorecard, tankClosure, p3_milestones, fa_priority, nodes, node_adj

    def merge_APdata(faro_scorecard, p3_milestones, tankClosure):
        ## organize milestone/tankClosure by LN
        tank_lookup = (
            tankClosure[['LINE_NUMBER', 'TankStatus']]
            .drop_duplicates(subset='LINE_NUMBER')
            .rename(columns={'LINE_NUMBER': 'LN'})
        )
        p3_lookup = (
            p3_milestones
            .rename(columns={'P': 'LN'})
            .copy()
        )

        ## merge all dataframes
        ap_df = (
            faro_scorecard
            .merge(tank_lookup, on='LN', how='left')
            .merge(p3_lookup, on='LN', how='left')
        )

        return ap_df

    faro_scorecard, tankClosure, p3_milestones, fa_priority, nodes, node_adj = import_data(rootpath, filepaths)
    ap_df = build_ap_df(faro_scorecard, p3_milestones, tankClosure)

    location_df = build_location_df(
        fa_priority=fa_priority,
        centerlines=CENTERLINES
    )

    labor_df = build_labor_df(
        fgi_staffing_byshift=FGI_STAFFING_BYSHIFT,
        fgi_cpj=FGI_CPJ,
        startdate=STARTDATE,
        enddate=ENDDATE
    )


    staffing_df = labor_df[labor_df['category'] == 'FGI_STAFFING_BYSHIFT'].copy()
    staffing_df['shift'] = staffing_df['shift'].astype(int)
    staffing_df['value'] = pd.to_numeric(staffing_df['value'])
    FGI_STAFFING_BYSHIFT = {
        shift: skill.set_index('team')['value'].to_dict()
        for shift, skill in staffing_df.groupby('shift', sort=True)
    }

    FGI_CPJ = (
        labor_df[labor_df['category'] == 'FGI_CPJ']
        .assign(value=lambda df: pd.to_numeric(df['value']))
        .set_index('team')['value']
        .to_dict()
    )



# build dataframes from FGI_LIVESTATE
if INPUT_TYPE == 'FGI_LIVESTATE':

    def load_live_state(path='FGI_liveState.xlsx'):
        required_columns = {
            'ap_data': [
                'LN', 'FA RO', 'FA RO to B1R', 'Total Counters', 'TankStatus', 'Ceilings',
                'Initial Tests Run', 'BTG_tot', 'BTG_p0', 'BTG_p1', 'BTG_p2', 'BTG_p3',
                'BTG_engines', 'BTG_doors', 'BTG_test', 'P3_Engine Hang',
                'P3_Flight Controls', 'P3_Gear Swing', 'P3_Medium Pressure Test',
                'P3_Oil On', 'P3_Power On', 'P3_Engine_Type',
                'P3_Milestone_Completion_Percentage', 'shakes_complete', 'shakes_req'
            ],
            'location_data': [
                'Location', 'Future State Priority', 'Date Online', 'Owner',
                'tooling_jacking', 'tooling_wings', 'tooling_tankClosure',
                'centerline_dependencies', 'obstructions', 'notes'
            ],
            'labor_data': ['category', 'shift', 'team', 'value']
        }

        numeric_ap_columns = [
            'LN', 'FA RO to B1R', 'Total Counters', 'TankStatus', 'Ceilings',
            'Initial Tests Run', 'BTG_tot', 'BTG_p0', 'BTG_p1', 'BTG_p2', 'BTG_p3',
            'BTG_engines', 'BTG_doors', 'BTG_test', 'P3_Engine Hang',
            'P3_Flight Controls', 'P3_Gear Swing', 'P3_Medium Pressure Test',
            'P3_Oil On', 'P3_Power On', 'P3_Milestone_Completion_Percentage',
            'shakes_complete', 'shakes_req'
        ]

        tooling_columns = ['tooling_jacking', 'tooling_wings', 'tooling_tankClosure']

        def _normalize_bool(value):
            if pd.isna(value):
                return False
            if isinstance(value, (bool, np.bool_)):
                return bool(value)
            if isinstance(value, (int, float)):
                return value != 0

            value_str = str(value).strip().lower()
            if value_str in {'true', 't', 'yes', 'y', '1'}:
                return True
            if value_str in {'false', 'f', 'no', 'n', '0', ''}:
                return False

            raise ValueError(f'Invalid boolean value in location_data: {value}')

        if not os.path.exists(path):
            raise ValueError(f'Live state workbook not found: {path}')

        try:
            workbook = pd.ExcelFile(path, engine='openpyxl')
        except FileNotFoundError as exc:
            raise ValueError(f'Live state workbook not found: {path}') from exc
        except Exception as exc:
            raise ValueError(f'Unable to open live state workbook: {path}') from exc

        missing_sheets = [sheet for sheet in required_columns if sheet not in workbook.sheet_names]
        if missing_sheets:
            raise ValueError(
                f"Missing required sheet(s) in {path}: {', '.join(missing_sheets)}"
            )

        ap_df = pd.read_excel(workbook, sheet_name='ap_data')
        location_df = pd.read_excel(workbook, sheet_name='location_data')
        labor_df = pd.read_excel(workbook, sheet_name='labor_data')

        frames = {
            'ap_data': ap_df,
            'location_data': location_df,
            'labor_data': labor_df
        }

        for sheet_name, expected_columns in required_columns.items():
            missing_columns = [col for col in expected_columns if col not in frames[sheet_name].columns]
            if missing_columns:
                raise ValueError(
                    f"Missing required column(s) in sheet '{sheet_name}': {', '.join(missing_columns)}"
                )

        ap_df = ap_df.copy()
        ap_df['FA RO'] = pd.to_datetime(ap_df['FA RO'], errors='coerce')
        for column in numeric_ap_columns:
            ap_df[column] = pd.to_numeric(ap_df[column], errors='coerce')

        if ap_df['LN'].notna().all():
            ap_df['LN'] = ap_df['LN'].astype('Int64')

        location_df = location_df.copy()
        location_df['Location'] = location_df['Location'].astype('string').str.strip()
        location_df = location_df[location_df['Location'].notna() & location_df['Location'].ne('')].reset_index(drop=True)
        for column in tooling_columns:
            location_df[column] = location_df[column].map(_normalize_bool).astype(bool)

        labor_df = labor_df.copy()
        labor_df['shift'] = pd.to_numeric(labor_df['shift'], errors='coerce').astype('Int64')

        return ap_df, location_df, labor_df
    
    ap_df, location_df, labor_df = load_live_state(path='FGI_liveState.xlsx')
    

if EXPORT_TO_FGI_LIVESTATE:

    output_path = EXPORT_PATH + 'FGI_liveState.xlsx'

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        ap_df.to_excel(writer, sheet_name='ap_data', index=False)
        location_df.to_excel(writer, sheet_name='location_data', index=False)
        labor_df.to_excel(writer, sheet_name='labor_data', index=False)

    if CODECELL_OUTPUT: print(output_path)

OSError: Cannot save file into a non-existent directory: '/output'

<p>|\<i>==> Now, dataframes have been built to describe APs, Locations, and Labor Allocation pararameters to build the schedule</i></p>

<h2>Scheduler</h2>
<h3>1. Class Definitions</h3>

In [ ]:
class AP:
    def __init__(self,LN,faro,toB1R,counters,btg=None,tankClosed=False,ceilings=0,testsRun=0,p3_milestones=None,shakes=None):
        self.LN=LN
        self.faro=faro
        self.toB1R=toB1R
        self.counters=counters
        self.btg=btg if btg is not None else {"tot":0,"p0":0,"p1":0,"p2":0,"p3":0,"engines":0,"doors":0,"test":0}
        self.tankClosed=tankClosed
        self.ceilings=ceilings
        self.testsRun=testsRun
        self.p3milestones=p3_milestones if p3_milestones is not None else {'tot':6,'complete':0}
        self.shakes = shakes if shakes is not None else {'complete': 0, 'req': 4}
        self.schedule={}
        self.moveReq=False
        self.path=[]
        self.Location = None
        self.taskState = 'RO'
        self.taskCompletion = {
            'compass': False,
            'paint': False,
        }

    # get FARO attributes
    def get_LN(self): return self.LN
    def get_FARO(self): return self.faro
    def get_toB1R(self): return self.toB1R
    def get_counters(self): return self.counters
    def get_BTG(self): return self.btg
    def get_tankClosure(self): return self.tankClosed
    def get_ceilings(self): return self.ceilings
    def get_testsRun(self): return self.testsRun
    def get_milestones(self): return self.milestones
    def get_schedule(self): return self.schedule
    def get_daystoB1R(self): return pd.Timedelta(days=self.toB1R)
    
    # move methods
    def isMoveReq(self): return self.moveReq
    def requireMove(self): 
        self.moveReq=True
        if CODECELL_OUTPUT: 
            print('=================MOVE REQUIRED==================')
            print(f'LN: {self.LN} requires move from {self.Location.get_name() if self.Location else "None"}')
            print('===============================================')

    # assignment methods
    def get_location(self): return self.Location
    def move(self, to_loc, date):
        self.Location = to_loc
        self.schedule[date] = to_loc.get_name()

    # scheduling methods
    def set_taskState(self, state):
        self.taskState = state

    def complete_BTG(self, category, btg_completed):
        self.btg[category] -= btg_completed

    # exit methods
    def exit(self,date):
        self.Location = None
        self.schedule[date] = 'B1R'
        if CODECELL_OUTPUT:
            print('=================AP EXIT==================')
            print(f'AP {self.LN} exited to B1R on {date.strftime("%Y-%m-%d")}')
            print(f'Path taken: {self.path + ["B1R"]}')
            print(f'FA RO on {self.faro.strftime("%Y-%m-%d")}, days to B1R: {self.toB1R}')
            print(f'Expected B1R date:{pd.to_timedelta(self.toB1R, unit="D")+ pd.to_datetime(self.faro).strftime("%Y-%m-%d")}')
            print('==========================================')

class Location:
    def __init__(self,priority,dateOnline,location,owner=None,tooling=None,obstructions=None,notes=None,centerline_dependencies=None):
        self.priority=priority
        self.dateOnline=dateOnline
        self.location=location
        self.owner=owner
        self.tooling=tooling if tooling is not None else {'jacking':False,'wings':False,'tankClosure':False}
        self.obstructions=obstructions if obstructions is not None else []
        self.notes=notes if notes is not None else []
        self.centerline_dependencies = centerline_dependencies
        self.schedule={}
        self.time_to = {}
        self.occupiedBy = None

    def get_priority(self): return self.priority
    def get_dateOnline(self): return self.dateOnline
    def get_name(self): return self.location
    def getName(self): return self.location
    def get_owner(self): return self.owner
    def get_tooling(self): return self.tooling
    def get_obstructions(self): return self.obstructions
    def get_notes(self): return self.notes
    def get_schedule(self): return self.schedule
    def get_occupiedBy(self): return self.occupiedBy
    def get_centerline_dependencies(self): return self.centerline_dependencies
    def canUse(self,tool): return self.tooling.get(tool,False)

    def isOnline(self):
        if self.dateOnline=='Now': return True
        if self.dateOnline=='At R10' and NEW_FA_ONLINE: return True
        return False

    def is_empty(self): return self.occupiedBy is None

    def assign(self, ap, date=None):
        self.occupiedBy = ap
    def unassign(self,date=None):
        self.occupiedBy = None



    def obstructs(self,obstruction): return obstruction in self.obstructions
    def isHigherPriorityThan(self,other): return self.priority>other.priority
    #def assign(self,ap,date): self.schedule[pd.to_datetime(date).normalize()]=ap
    def clear_schedule(self): self.schedule={}
    def set_time_to(self, other, move_time):
        self.time_to[other] = move_time

class FGI:
    def __init__(self, labor):
        self.labor = labor
        self.APs = {}
        self.Locations = {}
        self.locations = self.Locations
        self.queues = {
            'move': None,
            'FGI task': {
                'compass': [],
                'paint': []
            },
            'labor': {
                'structures': [],
                'system': [],
                'declam': [],
                'test': []
            }
        }
        self.schedule = {}
        self.chickenTracks = {}

    def get_queue(self, queue):
        if queue == 'all':
            return self.queues
        else:
            return self.queues[queue]

    def get_labor(self, shift, team):
        if team == 'all':
            return self.labor[shift]
        else:
            return self.labor[shift][team]
        
    def add_ap(self, LN, AP):
        self.APs[ln] = AP


    def add_Location(self, name, location_obj):
        self.Locations[name] = location_obj
        self.chickenTracks[name] = None

    def calibrateCompass(self, LN):
        self.APs[LN].taskState = 'CR3'
        self.APs[LN].requireMove()
        
        
    

    # def add_ap_toQueue(self, ln, ap=None, queues=None):
    #     target_queues = queues if queues is not None else ['structures', 'system', 'declam', 'test']
    #     for queue in target_queues:
    #         if ln not in self.queues[queue]:
    #             self.queues[queue].append(ln)
    #     # move queues are added explicitly later when the business rule requires movement

    # def add_location(self, name, loc_obj, availablity=True):
    #     self.Locations[name] = loc_obj
    #     self.locations_by_avail[availablity].append(name) if name not in self.locations_by_avail[availablity] else None

    # def is_location_available(self, loc):
    #     return loc in self.Locations and self.Locations[loc].is_empty()

    # def plan_move(self, ap, destination):
    #     self.queues['move'].append({
    #         'ap_ln': ap.get_LN(),
    #         'from': ap.current_location,
    #         'to': destination
    #     })

    # def get_move_time(self, i, j):
    #     return self.move_times.get(i, {}).get(j)

    # def update_location(self, ap, new_loc):
    #     old_loc = ap.location
    #     if old_loc in self.Locations:
    #         self.Locations[old_loc].unassign()
    #     if new_loc in self.Locations:
    #         self.Locations[new_loc].assign(ap)
    #     ap.location = new_loc
    #     ap.current_location = new_loc


SyntaxError: incomplete input (3224662977.py, line 196)

<h3>2. Function Definitions</h3>

<h4>a) Job Completion & Duration Helpers</h4>

In [ ]:
def get_fgi_btg(btg):
    fgi_tot = btg['tot'] - btg['p0'] - btg['p1']
    return {
        'fgi_tot': math.ceil(fgi_tot),
        'structure': math.ceil(0.1 * fgi_tot),
        'systems': math.ceil(btg['p2']),
        'declam': math.ceil(btg['p3'] + btg['engines']),
        'test': math.ceil(btg['test'])
    }

# def calc_btg_completion(fgi_staffing, cpj=FGI_CPJ, hours_per_shift=8):
#     manhours = {
#         shift: {team: staffing * hours_per_shift for team, staffing in teams.items()}
#         for shift, teams in fgi_staffing.items()
#     }
#     btg_completion = {
#         shift: {team: manhours[shift][team] / cpj[team] for team in teams}
#         for shift, teams in manhours.items()
#     }
#     return manhours, btg_completion


<h4>b) Scheduling Helpers</h4>

In [ ]:
def calibrate_compass(FGI, ln):
    ap = FGI.Locations[ln]
    ap.requireMove()
    
